In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import wilcoxon, mannwhitneyu, ttest_rel, ttest_ind
from statsmodels.stats.contingency_tables import mcnemar



In [5]:
pd.read_csv("../results/eegformer_5fold_run/subject_trial_stats_5fold.csv")
pd.read_csv("../results/hypothesis tests/cnn_lstm_eegnet_10seeds/all_trial_probs.csv")
pd.read_csv("../results/hypothesis tests/tgarnet_10seeds/all_trial_probs.csv")
pd.read_csv("../results/hypothesis tests/shallowconvnet_10seeds/all_trial_probs.csv")
pd.read_csv("../results/hypothesis tests/imcbgt_10seeds/all_trial_probs.csv")
pd.read_csv("../results/hypothesis tests/multistream_10seeds/all_trial_probs.csv")
pd.read_csv("../results/hypothesis tests/eegnet_10seeds/all_trial_probs.csv")

,seed,fold,subject,true_label,trial_global_idx,trial_local_idx,prob_adhd,pred_epoch,subject_majority_pred,subject_mean_prob
0,123,0,v107.mat,0,1934,0,0.000266,0,0,0.029382
1,123,0,v107.mat,0,1935,1,0.000953,0,0,0.029382
2,123,0,v107.mat,0,1936,2,0.027277,0,0,0.029382
3,123,0,v107.mat,0,1937,3,0.010203,0,0,0.029382
4,123,0,v107.mat,0,1938,4,0.001557,0,0,0.029382
...,...,...,...,...,...,...,...,...,...,...
16635,123,4,v60p.mat,0,3121,95,0.957561,1,1,0.840360
16636,123,4,v60p.mat,0,3122,96,0.959099,1,1,0.840360
16637,123,4,v60p.mat,0,3123,97,0.850334,1,1,0.840360
16638,123,4,v60p.mat,0,3124,98,0.841211,1,1,0.840360


In [15]:
# Analyze structure of the already-loaded CSV files using the same paths
csv_paths = {
    "EEGFormer": "../results/eegformer_5fold_run/all_trial_probs.csv",
    "CNN-LSTM-EEGNet": "../results/hypothesis tests/cnn_lstm_eegnet_10seeds/all_trial_probs.csv",
    "T-GarNet": "../results/hypothesis tests/tgarnet_10seeds/all_trial_probs.csv",
    "ShallowConvNet": "../results/hypothesis tests/shallowconvnet_10seeds/all_trial_probs.csv",
    "IMCBGT": "../results/hypothesis tests/imcbgt_10seeds/all_trial_probs.csv",
    "MultiStream": "../results/hypothesis tests/multistream_10seeds/all_trial_probs.csv",
    "EEGNet": "../results/hypothesis tests/eegnet_10seeds/all_trial_probs.csv",
}

loaded = {name: pd.read_csv(path) for name, path in csv_paths.items()}

rows = []
for name, df in loaded.items():
    cols = set(df.columns)
    level_tokens = []
    if any(c in cols for c in ["trial", "trial_id", "y_true", "y_pred", "pred_prob", "prob", "true_label"]):
        level_tokens.append("trial")
    if any(c in cols for c in ["subject", "subject_id", "subject_majority_pred"]):
        level_tokens.append("subject")
    if any(c in cols for c in ["fold", "fold_id"]):
        level_tokens.append("fold")
    if any(c in cols for c in ["seed", "run", "repeat"]):
        level_tokens.append("seed/repeat")

    rows.append(
        {
            "model": name,
            "n_rows": len(df),
            "n_cols": df.shape[1],
            "has_subject": "subject" in cols,
            "has_fold": "fold" in cols,
            "has_seed": "seed" in cols,
            "has_true_label": any(c in cols for c in ["label", "true_label", "y_true"]),
            "has_pred_label": any(c in cols for c in ["pred", "pred_label", "y_pred", "subject_majority_pred"]),
            "detected_levels": ", ".join(level_tokens) if level_tokens else "unknown",
            "key_columns_preview": ", ".join(sorted(df.columns)[:12]),
        }
    )

schema_summary = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
schema_summary

,model,n_rows,n_cols,has_subject,has_fold,has_seed,has_true_label,has_pred_label,detected_levels,key_columns_preview
0,CNN-LSTM-EEGNet,16640,10,True,True,True,True,True,"trial, subject, fold, seed/repeat","fold, pred_epoch, prob_adhd, seed, subject, su..."
1,EEGFormer,16640,10,True,True,True,True,True,"trial, subject, fold, seed/repeat","fold, pred_epoch, prob_adhd, seed, subject, su..."
2,EEGNet,16640,10,True,True,True,True,True,"trial, subject, fold, seed/repeat","fold, pred_epoch, prob_adhd, seed, subject, su..."
3,IMCBGT,16640,10,True,True,True,True,True,"trial, subject, fold, seed/repeat","fold, pred_epoch, prob_adhd, seed, subject, su..."
4,MultiStream,16640,10,True,True,True,True,True,"trial, subject, fold, seed/repeat","fold, pred_epoch, prob_adhd, seed, subject, su..."
5,ShallowConvNet,16640,10,True,True,True,True,True,"trial, subject, fold, seed/repeat","fold, pred_epoch, prob_adhd, seed, subject, su..."
6,T-GarNet,16640,10,True,True,True,True,True,"trial, subject, fold, seed/repeat","fold, pred_epoch, prob_adhd, seed, subject, su..."


In [7]:
# Exact column-level inspection for interpretation
for model_name, df in loaded.items():
    print(f"\n[{model_name}] columns ({len(df.columns)}):")
    print(list(df.columns))


[EEGFormer] columns (8):
['subject', 'trial_pct_class1', 'n_trials', 'n_class1', 'mean_prob', 'label', 'n_class0', 'trial_pct_class0']

[CNN-LSTM-EEGNet] columns (10):
['seed', 'fold', 'subject', 'true_label', 'trial_global_idx', 'trial_local_idx', 'prob_adhd', 'pred_epoch', 'subject_majority_pred', 'subject_mean_prob']

[T-GarNet] columns (10):
['seed', 'fold', 'subject', 'true_label', 'trial_global_idx', 'trial_local_idx', 'prob_adhd', 'pred_epoch', 'subject_majority_pred', 'subject_mean_prob']

[ShallowConvNet] columns (10):
['seed', 'fold', 'subject', 'true_label', 'trial_global_idx', 'trial_local_idx', 'prob_adhd', 'pred_epoch', 'subject_majority_pred', 'subject_mean_prob']

[IMCBGT] columns (10):
['seed', 'fold', 'subject', 'true_label', 'trial_global_idx', 'trial_local_idx', 'prob_adhd', 'pred_epoch', 'subject_majority_pred', 'subject_mean_prob']

[MultiStream] columns (10):
['seed', 'fold', 'subject', 'true_label', 'trial_global_idx', 'trial_local_idx', 'prob_adhd', 'pred_epoc

In [8]:
# Count key units for each model (subjects/folds/seeds/trials)
count_rows = []
for name, df in loaded.items():
    row = {"model": name, "rows": len(df)}
    for col in ["subject", "fold", "seed", "trial_global_idx", "trial_local_idx"]:
        row[f"n_{col}"] = int(df[col].nunique()) if col in df.columns else np.nan
    count_rows.append(row)

pd.DataFrame(count_rows).sort_values("model").reset_index(drop=True)

,model,rows,n_subject,n_fold,n_seed,n_trial_global_idx,n_trial_local_idx
0,CNN-LSTM-EEGNet,16640,120,5.0,1.0,3710.0,336.0
1,EEGFormer,120,120,NaN,NaN,NaN,NaN
2,EEGNet,16640,120,5.0,1.0,3710.0,336.0
3,IMCBGT,16640,120,5.0,1.0,3710.0,336.0
4,MultiStream,16640,120,5.0,1.0,3710.0,336.0
5,ShallowConvNet,16640,120,5.0,1.0,3710.0,336.0
6,T-GarNet,16640,120,5.0,1.0,3710.0,336.0


In [13]:
# Verify trial/fold/subject alignment across all models
base = loaded["EEGFormer"].copy()
key_cols = ["seed", "fold", "subject", "trial_global_idx", "trial_local_idx"]

base_keys = set(map(tuple, base[key_cols].to_numpy()))
base_subjects = set(base["subject"].unique())
base_subject_fold = set(map(tuple, base[["subject", "fold"]].drop_duplicates().to_numpy()))

alignment_rows = []
for model, df in loaded.items():
    model_keys = set(map(tuple, df[key_cols].to_numpy()))
    model_subjects = set(df["subject"].unique())
    model_subject_fold = set(map(tuple, df[["subject", "fold"]].drop_duplicates().to_numpy()))

    alignment_rows.append(
        {
            "model": model,
            "rows": len(df),
            "same_trial_keys_as_eegformer": model_keys == base_keys,
            "missing_trial_keys_vs_eegformer": len(base_keys - model_keys),
            "extra_trial_keys_vs_eegformer": len(model_keys - base_keys),
            "same_subjects_as_eegformer": model_subjects == base_subjects,
            "same_subject_fold_pairs_as_eegformer": model_subject_fold == base_subject_fold,
            "n_subjects": len(model_subjects),
            "n_subject_fold_pairs": len(model_subject_fold),
        }
    )

alignment_summary = pd.DataFrame(alignment_rows).sort_values("model").reset_index(drop=True)
alignment_summary

,model,rows,same_trial_keys_as_eegformer,missing_trial_keys_vs_eegformer,extra_trial_keys_vs_eegformer,same_subjects_as_eegformer,same_subject_fold_pairs_as_eegformer,n_subjects,n_subject_fold_pairs
0,CNN-LSTM-EEGNet,16640,True,0,0,True,True,120,120
1,EEGFormer,16640,True,0,0,True,True,120,120
2,EEGNet,16640,True,0,0,True,True,120,120
3,IMCBGT,16640,True,0,0,True,True,120,120
4,MultiStream,16640,True,0,0,True,True,120,120
5,ShallowConvNet,16640,True,0,0,True,True,120,120
6,T-GarNet,16640,True,0,0,True,True,120,120


In [14]:
# Fold composition check (important for fold-level inference)
fold_subject_counts = (
    loaded["EEGFormer"][["fold", "subject"]]
    .drop_duplicates()
    .groupby("fold")
    .size()
    .rename("n_subjects")
    .reset_index()
)
fold_subject_counts

,fold,n_subjects
0,0,24
1,1,24
2,2,24
3,3,24
4,4,24


## Updated test insight after alignment check

Alignment result from current `csv_paths`:
- All models have identical trial keys (`seed`, `fold`, `subject`, `trial_global_idx`, `trial_local_idx`).
- All models share the same 120 subjects.
- Fold split is balanced: 5 folds x 24 subjects per fold.

### Trial-level comparison (matched trials)
- Unit is paired at trial level across models.
- Appropriate options:
  - Paired binary disagreement test (McNemar-style) **with subject-cluster handling**.
  - Preferred robust approach: clustered bootstrap/permutation by subject, or GEE logistic with subject as cluster.
- Avoid naive independent trial tests (plain z-test / independent t-test), because within-subject trial correlation violates independence.

### Subject-level comparison (matched subjects)
- Build one subject prediction per model (e.g., subject majority prediction).
- Use **exact McNemar** per pair: EEGFormer vs each baseline (primary hypothesis test).
- Report discordant counts, p-value, and paired effect size (accuracy delta with CI).

### Fold-level comparison (matched folds)
- Compute fold metrics for each model (balanced accuracy, F1, ROC-AUC).
- Compare EEGFormer vs baseline with **paired Wilcoxon signed-rank** over 5 paired folds.
- With only 5 folds, also report effect sizes and per-fold deltas; treat p-values as low-power evidence.

### Practical reporting order
1. Primary endpoint: subject-level McNemar (most interpretable clinically).
2. Secondary: fold-level metric deltas + paired Wilcoxon.
3. Exploratory: trial-level clustered model (GEE/mixed effects).
4. Multiple testing correction across 6 model comparisons: Holm-Bonferroni.

## Practical statistical plan for EEGFormer vs baselines

### 1) Data structure in the loaded CSVs

- **Proposed model**: EEGFormer (`subject_trial_stats_5fold.csv`).
- **Baselines/comparators**: CNN-LSTM-EEGNet, T-GarNet, ShallowConvNet, IMCBGT, MultiStream, EEGNet (their `all_trial_probs.csv` files).

Observed levels:
- **Trial level** (baselines): `trial_global_idx`, `trial_local_idx`, `prob_adhd`, `pred_epoch`, `true_label`, with repeated observations within subject and fold.
- **Subject level** (all models): `subject`, ground-truth label (`label`/`true_label`), and subject-level prediction signal (`subject_majority_pred` or EEGFormer majority from `n_class1` and `n_trials`).
- **Fold level** (baselines only in currently loaded files): `fold` is explicit. EEGFormer file here is already aggregated at subject level.

### 2) Which tests are appropriate and why

Use **paired tests** whenever comparing EEGFormer to another model on the same subjects/folds.

- **Primary subject-level binary endpoint** (correct/incorrect per subject):
  - Use **exact McNemar test** for each pair: EEGFormer vs comparator.
  - Why: paired nominal outcomes on the same subjects; directly tests disagreement counts.

- **Fold-level metric comparison** (if you build fold-wise metrics per model, e.g., balanced accuracy, F1, ROC-AUC):
  - Use **Wilcoxon signed-rank** on paired fold scores (EEGFormer vs comparator).
  - Why: paired repeated folds, non-normality-safe with small `k=5`.
  - Avoid independent tests (`mannwhitneyu`, `ttest_ind`) because folds are matched, not independent groups.

- **Trial-level analysis**:
  - Do **not** use plain proportion z-tests on raw trials (independence violated due to clustering within subject/fold).
  - Preferred: **cluster-aware model**, e.g., GEE logistic or mixed-effects logistic with subject as cluster/random effect and model as fixed effect.

### 3) Metrics to compare

Recommended reporting levels:
- **Primary**: subject-level accuracy (or balanced accuracy) with McNemar p-values.
- **Secondary**: fold-level balanced accuracy, F1, ROC-AUC (paired Wilcoxon + effect size).
- **Calibration/probability** (optional): subject mean probability error/Brier score if probability quality matters.

### 4) Significance reporting in a paper

- State one primary endpoint and one primary test before analysis.
- Report: effect size + 95% CI + p-value (not p-value alone).
- Correct for multiple pairwise comparisons across 6 baselines:
  - Prefer **Holm-Bonferroni** (or Benjamini-Hochberg if controlling FDR).
- For McNemar, report discordant counts `(EEGFormer correct, baseline wrong)` and `(EEGFormer wrong, baseline correct)`.
- Include practical effect magnitude (absolute accuracy delta in percentage points).

### 5) Important limitation from current loaded paths

- The loaded EEGFormer file is subject-level aggregated. For fully aligned fold/trial paired testing against baseline `all_trial_probs.csv`, also load EEGFormer trial/fold predictions (or a fold-wise EEGFormer file) so all models are compared on the same unit of analysis.

## Update: best EEGFormer file for hypothesis testing

Use `../results/eegformer_5fold_run/all_trial_probs.csv` as the primary EEGFormer file for hypothesis tests.

Why this is better:
- It has the same core structure as baseline `all_trial_probs.csv` files (`seed`, `fold`, `subject`, `true_label`, trial-level probabilities/predictions).
- It enables **unit-matched paired comparisons** at trial, subject, and fold levels.

Recommended practical workflow:
1. Build per-subject correctness per fold for each model -> pair with EEGFormer -> exact McNemar (primary).
2. Build per-fold metrics (balanced accuracy, F1, ROC-AUC) per model -> paired Wilcoxon vs EEGFormer.
3. If using trial-level outcomes directly, use clustered methods (GEE / mixed-effects logistic), not naive independent-proportion tests.

In [9]:
# Load EEGFormer trial-level file for aligned model-vs-model tests
eegformer_trials = pd.read_csv("../results/eegformer_5fold_run/all_trial_probs.csv")

print("EEGFormer trial file shape:", eegformer_trials.shape)
print("Columns:", list(eegformer_trials.columns))
print("Unique subjects/folds/seeds:",
      eegformer_trials["subject"].nunique(),
      eegformer_trials["fold"].nunique(),
      eegformer_trials["seed"].nunique())

eegformer_trials.head()

EEGFormer trial file shape: (16640, 10)
Columns: ['seed', 'fold', 'subject', 'true_label', 'trial_global_idx', 'trial_local_idx', 'prob_adhd', 'pred_epoch', 'subject_majority_pred', 'subject_mean_prob']
Unique subjects/folds/seeds: 120 5 1


,seed,fold,subject,true_label,trial_global_idx,trial_local_idx,prob_adhd,pred_epoch,subject_majority_pred,subject_mean_prob
0,123,0,v107.mat,0,1934,0,0.001627,0,0,0.036487
1,123,0,v107.mat,0,1935,1,0.000845,0,0,0.036487
2,123,0,v107.mat,0,1936,2,0.000228,0,0,0.036487
3,123,0,v107.mat,0,1937,3,0.000031,0,0,0.036487
4,123,0,v107.mat,0,1938,4,0.000820,0,0,0.036487


## EEGFormer data

In [2]:
eegformer_data = pd.read_csv("../results/eegformer_5fold_run/subject_trial_stats_5fold.csv")
eegformer_data

,subject,trial_pct_class1,n_trials,n_class1,mean_prob,label,n_class0,trial_pct_class0
0,v179.mat,100.0,98,98,9.997370e-01,1,0,0.0
1,v213.mat,100.0,95,95,9.999760e-01,1,0,0.0
2,v21p.mat,100.0,128,128,9.998449e-01,1,0,0.0
3,v227.mat,100.0,218,218,9.998041e-01,1,0,0.0
4,v22p.mat,100.0,93,93,9.987505e-01,1,0,0.0
...,...,...,...,...,...,...,...,...
115,v125.mat,0.0,119,0,1.101914e-07,0,119,100.0
116,v131.mat,0.0,129,0,1.813123e-04,0,129,100.0
117,v133.mat,0.0,116,0,3.938721e-06,0,116,100.0
118,v140.mat,0.0,133,0,1.395331e-03,0,133,100.0


In [145]:
def build_eegformer_subject_majority(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["label"] = out["label"].astype(int)
    out["n_trials"] = out["n_trials"].astype(int)
    out["n_class1"] = out["n_class1"].astype(int)
    out["pred_majority"] = (out["n_class1"] >= (out["n_trials"] / 2.0)).astype(int)
    out["correct"] = (out["pred_majority"] == out["label"]).astype(int)
    return out[["subject", "label", "pred_majority", "correct"]].copy()

eegformer_subj = build_eegformer_subject_majority(eegformer_data)
eegformer_subj.head()

,subject,label,pred_majority,correct
0,v179.mat,1,1,1
1,v213.mat,1,1,1
2,v21p.mat,1,1,1
3,v227.mat,1,1,1
4,v22p.mat,1,1,1


## CNN LSTM EEGNet Data

In [3]:
cnn_lstm_eegnet_data = pd.read_csv("../results/hypothesis tests/cnn_lstm_eegnet_10seeds/all_trial_probs.csv")
cnn_lstm_eegnet_data.head()

,seed,fold,subject,true_label,trial_global_idx,trial_local_idx,prob_adhd,pred_epoch,subject_majority_pred,subject_mean_prob
0,123,0,v107.mat,0,1934,0,0.000009,0,0,0.011979
1,123,0,v107.mat,0,1935,1,0.000020,0,0,0.011979
2,123,0,v107.mat,0,1936,2,0.000014,0,0,0.011979
3,123,0,v107.mat,0,1937,3,0.000023,0,0,0.011979
4,123,0,v107.mat,0,1938,4,0.000043,0,0,0.011979


In [147]:
def build_cnlstm_subject_majority(df: pd.DataFrame) -> pd.DataFrame:
    needed = {"subject", "true_label", "subject_majority_pred"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns for CNLSTM data: {sorted(missing)}")

    out = (
        df.groupby("subject", as_index=False)
        .agg(label=("true_label", "first"), pred_majority=("subject_majority_pred", "first"))
        .copy()
    )
    out["label"] = out["label"].astype(int)
    out["pred_majority"] = out["pred_majority"].astype(int)
    out["correct"] = (out["pred_majority"] == out["label"]).astype(int)
    return out


cnlstm_subj = build_cnlstm_subject_majority(cnn_lstm_eegnet_data)
cnlstm_subj.head()

,subject,label,pred_majority,correct
0,v107.mat,0,0,1
1,v108.mat,0,0,1
2,v109.mat,0,0,1
3,v10p.mat,1,1,1
4,v110.mat,0,0,1


## T-GarNet

In [155]:
tgarnet_data = pd.read_csv("../results/hypothesis tests/tgarnet_10seeds/all_trial_probs.csv")
tgarnet_data.head()

FileNotFoundError: [Errno 2] No such file or directory: '../results/hypothesis tests/tgarnet_10seeds/all_trial_probs.csv'

In [148]:
data_subj = (
    eegformer_subj[["subject", "correct"]]
    .rename(columns={"correct": "EEGFormer"})
    .merge(
        cnlstm_subj[["subject", "correct"]].rename(columns={"correct": "CNLSTM"}),
        on="subject",
        how="inner",
    )
    .sort_values("subject")
    .reset_index(drop=True)
)

data_subj

,subject,EEGFormer,CNLSTM
0,v107.mat,1,1
1,v108.mat,1,1
2,v109.mat,1,1
3,v10p.mat,1,1
4,v110.mat,1,1
...,...,...,...
115,v58p.mat,1,1
116,v59p.mat,0,0
117,v60p.mat,0,0
118,v6p.mat,1,1


## McNemar test

In [153]:
both_correct = int(((data_subj["EEGFormer"] == 1) & (data_subj["CNLSTM"] == 1)).sum())
eegformer_only = int(((data_subj["EEGFormer"] == 1) & (data_subj["CNLSTM"] == 0)).sum())
cnlstm_only = int(((data_subj["EEGFormer"] == 0) & (data_subj["CNLSTM"] == 1)).sum())
both_wrong = int(((data_subj["EEGFormer"] == 0) & (data_subj["CNLSTM"] == 0)).sum())

mcnemar_table = [
    [both_correct, eegformer_only],
    [cnlstm_only, both_wrong],
]

result_mcnemar = mcnemar(mcnemar_table, exact=True)

print("McNemar contingency table:")
print(f"[[{both_correct}, {eegformer_only}], [{cnlstm_only}, {both_wrong}]]")
print(f"McNemar statistic: {result_mcnemar.statistic:.4f}")
print(f"P-value: {result_mcnemar.pvalue:.3%}")

McNemar contingency table:
[[94, 11], [2, 13]]
McNemar statistic: 2.0000
P-value: 2.246%


## Proportion test

In [149]:
count_eegformer = int(data_subj["EEGFormer"].sum())
count_cnn_lstm_eegnet = int(data_subj["CNLSTM"].sum())
nobs_eegformer = int(len(data_subj))
nobs_cnn_lstm_eegnet = int(len(data_subj))

print(f"EEGFormer: {count_eegformer} correct out of {nobs_eegformer} subjects")
print(f"CNN-LSTM-EEGNet: {count_cnn_lstm_eegnet} correct out of {nobs_cnn_lstm_eegnet} subjects")

z_stat, p_ztest = proportions_ztest(
    count=[count_eegformer, count_cnn_lstm_eegnet],
    nobs=[nobs_eegformer, nobs_cnn_lstm_eegnet],
    alternative="larger",
)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_ztest:.3%}")

EEGFormer: 105 correct out of 120 subjects
CNN-LSTM-EEGNet: 96 correct out of 120 subjects
Z-statistic: 1.5748
P-value: 5.765%


## Wilconxon

In [150]:
stat, p = wilcoxon(
    data_subj["EEGFormer"],
    data_subj["CNLSTM"],
    alternative="greater",
)

print(f"Wilcoxon signed-rank test statistic: {stat:.4f}")
print(f"Wilcoxon signed-rank test p-value: {p:.3%}")

Wilcoxon signed-rank test statistic: 77.0000
Wilcoxon signed-rank test p-value: 0.628%


## MannWithneyU

In [151]:
U_stat, p_mannwhitney = mannwhitneyu(
    data_subj["EEGFormer"],
    data_subj["CNLSTM"],
    alternative="greater",
)

print(f"Mann-Whitney U statistic: {U_stat:.4f}")
print(f"P-value: {p_mannwhitney:.3%}")

Mann-Whitney U statistic: 7740.0000
P-value: 5.820%


## T-tests

In [152]:
t_stat_ind, p_ttest_ind = ttest_ind(
    data_subj["EEGFormer"],
    data_subj["CNLSTM"],
    alternative="greater",
)
print(f"Independent t-test statistic: {t_stat_ind:.4f}")
print(f"P-value: {p_ttest_ind:.3%}")

t_stat_rel, p_ttest_rel = ttest_rel(
    data_subj["EEGFormer"],
    data_subj["CNLSTM"],
    alternative="greater",
)
print(f"Paired t-test statistic: {t_stat_rel:.4f}")
print(f"P-value: {p_ttest_rel:.3%}")

Independent t-test statistic: 1.5764
P-value: 5.814%
Paired t-test statistic: 2.5529
P-value: 0.597%
